# 03 — CNN para Classificação de EEG

Este notebook treina uma CNN 2D para classificar estados mentais a partir das features espectrais do EEG.

A entrada da rede é uma grade **5×5×N**, onde:
- **Linhas** = canais EEG (AF3, AF4, T7, T8, Pz)
- **Colunas** = bandas de frequência (Theta, Alpha, Beta, Gamma, ...)
- **Profundidade (canais CNN)** = N frames temporais consecutivos empilhados

O janelamento temporal empilha N=3 amostras consecutivas como canais, dando à CNN contexto de como o sinal EEG evoluiu ao longo do tempo — principal diferença em relação à versão sem contexto temporal.

Os dados utilizados são os gerados pelo notebook `02_preprocessamento_base.ipynb` com `split_strategy="random"`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

from src.models.cnn import CNNConfig, train_cnn

## Carregar dados pré-processados

In [ ]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

X_train = np.load(PROCESSED_DIR / "X_train_base.npy")
X_test  = np.load(PROCESSED_DIR / "X_test_base.npy")
y_train = np.load(PROCESSED_DIR / "y_train.npy")
y_test  = np.load(PROCESSED_DIR / "y_test.npy")

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)

## Treinar o modelo

In [ ]:
config = CNNConfig()

result = train_cnn(X_train, y_train, X_test, y_test, config)

## Curvas de aprendizado

Os gráficos abaixo mostram como a loss e a acurácia evoluíram durante o treino.
- A linha **azul** (train) representa o desempenho nos dados de treino.
- A linha **laranja** (val) representa o desempenho nos dados de validação (20% do treino).

Se as duas linhas ficarem muito separadas, o modelo está decorando os dados de treino (overfitting).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs_ran = range(1, len(result.history["loss"]) + 1)

ax1.plot(epochs_ran, result.history["loss"], label="Treino")
ax1.plot(epochs_ran, result.history["val_loss"], label="Validação")
ax1.set_title("Loss")
ax1.set_xlabel("Época")
ax1.set_ylabel("Loss")
ax1.legend()

ax2.plot(epochs_ran, result.history["accuracy"], label="Treino")
ax2.plot(epochs_ran, result.history["val_accuracy"], label="Validação")
ax2.set_title("Acurácia")
ax2.set_xlabel("Época")
ax2.set_ylabel("Acurácia")
ax2.legend()

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "figures" / "cnn_learning_curves.png", dpi=150)
plt.show()

## Resultado no conjunto de teste

In [ ]:
print(f"Loss no teste:     {result.test_loss:.4f}")
print(f"Acurácia no teste: {result.test_accuracy:.4f} ({result.test_accuracy * 100:.2f}%)")

## Matriz de confusão

Mostra quantas amostras de cada classe foram classificadas corretamente e quais foram confundidas com outras classes.

In [ ]:
cm = confusion_matrix(result.y_test, result.y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Esquerda", "Direita", "Neutro"])
disp.plot(ax=ax, colorbar=False)
ax.set_title("Matriz de Confusão — CNN")

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "figures" / "cnn_confusion_matrix.png", dpi=150)
plt.show()

## Relatório de classificação

- **Precision**: de tudo que o modelo disse ser classe X, quantos eram de fato classe X.
- **Recall**: de todas as amostras da classe X, quantas o modelo acertou.
- **F1-score**: média harmônica entre precision e recall — a métrica mais equilibrada para comparar modelos.

In [ ]:
print(classification_report(result.y_test, result.y_pred, target_names=["Esquerda", "Direita", "Neutro"]))

## Salvar modelo

In [ ]:
MODELS_DIR = PROJECT_ROOT / "outputs" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

result.model.save(MODELS_DIR / "cnn_model.keras")
print("Modelo salvo em outputs/models/cnn_model.keras")

## Exportar métricas para o dashboard

In [ ]:
from src.evaluation.export_metrics import export_metrics

users_test = np.load(PROCESSED_DIR / "users_test.npy", allow_pickle=True)
export_metrics(result, model_name="cnn", users_test=users_test)